In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sn
# import pytesseract as tb

import csv

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:,.2f}'.format
sn.set_style('whitegrid')
%matplotlib inline

pathh = "./data/intermediate/spain/descriptive_stats/"

In [17]:
def format_log_files(files,path,filame_target,keywords_start=["tab","tabstat"],keywords_end=["N","Total"],
                     loop_tab=False,descriptive_var_list=None,debug=False):
    table_no = 1
    if loop_tab:
        index_key = 0
    else:
        index_key = 1
    for file in files:
        print(file)
        res_spa_file = path+file
        with open(res_spa_file) as file:
            reading = 0
            type_i = 0
            lines = file.read().splitlines()
            output = []
            for line in lines:
                line = line.replace('|','')
                words = line.split()
                if debug:
                    print(words)
                if len(words)>=1 and words[index_key] in keywords_start :
                    # print()
                    reading = 1
                elif len(words)>2 and words[0] not in ['.','opened','','closed'] and reading:
                    if descriptive_var_list!=None:
                        if words[0] in descriptive_var_list:
                            output = []
                            row_length = len(words)
                            # print(row_length)
                    filterr = 1 - np.array([i.replace(",","").replace(".","").isnumeric() for i in words])
        
                    if descriptive_var_list!=None:
                        if words[0] in descriptive_var_list:
                            words_filtered = words                       
                        elif filterr.sum()==0 or np.sum(1-filterr)==0:
                            words_filtered = words
                        else:                    
                            var = " ".join(words[:filterr.sum()])                
                            words_filtered = [var,] + [i.replace(",","") for i in words[filterr.sum():]]

                    else:
                        if filterr.sum()==0 or np.sum(1-filterr)==0:
                            words_filtered = words
                        else:                    
                            var = " ".join(words[:filterr.sum()])                
                            words_filtered = [var,] + [i.replace(",","") for i in words[filterr.sum():]]
                    
                    if descriptive_var_list!=None:
                        if len(words_filtered) == row_length-1:
                            words_filtered = [' ',] + words_filtered
                    output.append(words_filtered)
                    if words[0] in keywords_end:
                        # print(output, table_no)
                        reading=0
                        with open(path+filame_target+"{}.csv".format(table_no), 'w',newline='') as csvfile:
                            w = csv.writer(csvfile)
                            w.writerows(output)
                            table_no += 1
                            output = []
    print('Done!')

<a id="index"></a>

# Index

- [Duration of unstable spells (tab:desstats_dur)](#dur)
- [Main descriptive stats – individual level (tab:desstats_2+spell)](#main)
- [Education breakdown (tab:desstats_2+spell)](#educ)


In [18]:
res_ger_file = r'./data/intermediate/germany/05_descriptives.log'


In [19]:
with open(res_ger_file) as file:
    reading = 0
    type_i = 0
    table_no = 1
    row_length = None
    lines = file.read().splitlines()
    output = []
    descriptive_var_list = ['attached','unattached','max_TC_sample','Industry','educ2','state_detail','state_c',"Employment"]

    for line in lines:
        line = line.replace('|','')
        words = line.split()

        if len(words) > 1 and (words[1] == "tab" or words[1] == "tabstat") and words[0] != ">":
            reading = 1
            row_length = None

        elif len(words) > 2 and words[0] not in ['.','opened','','closed'] and words[1] != '' and reading:
            if words[0] in descriptive_var_list:
                output = []
                if words[0] == "Employment":
                    man_i = words.pop(1)
                    words[0] = words[0] + " " + man_i
                    descriptive_var_list[-1] = words[0]
                row_length = len(words)

            filterr = 1 - np.array([i.replace(",","").replace(".","").isnumeric() for i in words])

            if words[0] in descriptive_var_list:
                words_filtered = words
            elif filterr.sum() == 0 or np.sum(1 - filterr) == 0:
                words_filtered = words
            else:
                var = " ".join(words[:filterr.sum()])
                words_filtered = [var] + [i.replace(",","") for i in words[filterr.sum():]]
                if words[0] == "Manufacturing":
                    man_i = words_filtered.pop(1)
                    words_filtered[0] = words_filtered[0] + " " + man_i

            if row_length is not None and len(words_filtered) == row_length - 1:
                words_filtered = [' '] + words_filtered

            output.append(words_filtered)

            if words[0] == 'N' or words[0] == 'Total':
                reading = 0
                with open("./output/intermediate/descriptive_stats_ger_tab{}.csv".format(table_no), 'w', newline='') as csvfile:
                    w = csv.writer(csvfile)
                    w.writerows(output)
                    table_no += 1
                    output = []


For Spain, we need to format table by table:

In [20]:
# files = ["desstats_all_id_basic.log","desstats_all_id_educ.log", "desstats_all_id_ind.log", 
#          "desstats_sple_spells_basic.log","desstats_sple_id_basic.log"]
# target_name = "descriptive_stats_spa_tab"

file_descriptive = ["desstats_sple_id_basic2011_classic.log"]
target_name = "descriptive_stats_spa_tab_sample"

format_log_files(file_descriptive,pathh,target_name,keywords_start=["Group"],keywords_end=["N","Total"],
                 descriptive_var_list=['max_TC_sample_classic'],loop_tab=True)

files_new = ["desstats_sple_id_educ2011_classic.log"]
target_name = "descriptive_stats_spa_other_sample"

format_log_files(files_new,pathh,target_name,keywords_start=["Group",'max_TC_sample_classic'],keywords_end=["N","Total"],
                 descriptive_var_list=["educ2"],loop_tab=True)

desstats_sple_id_basic2011_classic.log
Done!
desstats_sple_id_educ2011_classic.log
Done!


<a id="dur"></a>

### 0. Duration of unstable spells (`tab:desstats_dur`)

**Mean and Log(variance)** come from the baseline decomposition result CSVs
(`mu_y` and `lsig_y2` fields):
- Germany: `archive/germany/Results May 2025/031_decomp.csv`
- Spain:   `data/intermediate/spain/results/results2011_T_harm.csv`

**Variance and percentiles** come from the Stata tabstat logs (spell-level,
`max_TC_sample==2`):
- Germany: `archive/germany/Results May 2025/05_descriptives.log` Table 9
- Spain:   `data/intermediate/spain/descriptive_stats/desstats_sple_spells_states2011.log`

**N** = individual count: Germany 71,046 (from Stata tab4); Spain = `Obs/2` from decomp CSV.

*Note: Spain numbers differ slightly from the paper because the paper used
the v20 baseline data; the current files reflect the v21 (2021) update.*


In [21]:
## Mean (mu_y) and Log(variance) (lsig_y2) from decomposition result CSVs
decomp_ger = pd.read_csv(
    "./data/intermediate/germany/031_decomp_2011.csv", index_col=0)
decomp_spa = pd.read_csv(
    "./data/intermediate/spain/results/results2011_T_harm.csv", index_col=0)

ger_mu_y    = decomp_ger.loc["mu_y",    "Mean"]
ger_lsig_y2 = decomp_ger.loc["lsig_y2", "Mean"]
ger_N_ind   = int(decomp_ger.loc["lsig_y2", "Obs"].replace(",", "")) // 2

spa_mu_y    = decomp_spa.loc["mu_y",    "Mean"]
spa_lsig_y2 = decomp_spa.loc["lsig_y2", "Mean"]
spa_N_ind   = int(decomp_spa.loc["lsig_y2", "Obs"].replace(",", "")) // 2

print(f"Germany: mu_y={ger_mu_y:.3f}, lsig_y2={ger_lsig_y2:.3f}, N={ger_N_ind}")
print(f"Spain:   mu_y={spa_mu_y:.3f}, lsig_y2={spa_lsig_y2:.3f}, N={spa_N_ind}")


Germany: mu_y=360.175, lsig_y2=2.568, N=81768
Spain:   mu_y=912.447, lsig_y2=1.855, N=69406


In [22]:
## Helper: parse a "tabstat X Y, by(max_TC_sample)" block from a Stata log
def parse_tabstat_dur(log_path, start_check_fn, end_check_fn):
    """Return dict {group_id: [[val1,val2], ...]} with 10 stat rows per group."""
    stats_order = ["Mean","SD","Min","p10","p25","p50","p75","p90","Max","N"]
    reading, group_id, rows, result = False, None, [], {}
    with open(log_path) as f:
        lines = f.read().splitlines()
    for line in lines:
        lc    = line.replace("|", "")
        words = lc.split()
        bare  = [w.rstrip(",;") for w in words]
        if not words:
            continue
        if not reading and start_check_fn(bare):
            reading, group_id, rows = True, None, []
            continue
        if reading and end_check_fn(bare, line):
            if group_id is not None:
                result[group_id] = rows
            reading = False
        if not reading:
            continue
        if words[0] in ["Summary","Group","max_TC_sample"]:
            continue
        if all(c in "-+| " for c in lc):
            continue
        if words[0] == "Total":
            if group_id is not None:
                result[group_id] = rows
            group_id, rows = "Total", []
            continue
        try:
            vals = [float(w.replace(",","")) for w in words[-2:]]
        except ValueError:
            continue
        if len(words) >= 3:
            if group_id is not None:
                result[group_id] = rows
            group_id, rows = words[0], [vals]
        elif len(words) == 2:
            rows.append(vals)
    return stats_order, result

## Germany Table 9: ". tabstat days_if_P days_if_TU, by(max_TC_sample) ..."
stats_order, ger_dur = parse_tabstat_dur(
    res_ger_file,
    start_check_fn=lambda b: (
        len(b)>=3 and b[0]=="." and b[1]=="tabstat"
        and "days_if_P" in b and "days_if_TU" in b),
    end_check_fn=lambda b, line: (
        b[0]=="." and len(b)>=2 and b[1]!=">")
)
ger_g2 = {stats_order[i]: ger_dur["2"][i] for i in range(len(stats_order))}
tab_ger_dur = pd.DataFrame(ger_g2, index=["days_if_P","days_if_TU"]).T

## Spain log: same tabstat from Descriptive_stats_main.do
stats_order, spa_dur = parse_tabstat_dur(
    pathh + "desstats_sple_spells_states2011.log",
    start_check_fn=lambda b: (
        len(b)>=2 and b[1]=="tabstat" and "days_if_TU" in b),
    end_check_fn=lambda b, line: "log close" in line
)
spa_g2 = {stats_order[i]: spa_dur["2"][i] for i in range(len(stats_order))}
tab_spa_dur = pd.DataFrame(spa_g2, index=["days_if_P","days_if_TU"]).T

tab_ger_dur


,days_if_P,days_if_TU
Mean,550.48,560.77
SD,668.76,643.53
Min,1.00,0.00
p10,29.00,29.00
p25,66.00,88.00
p50,277.00,317.00
p75,777.00,819.00
p90,"1,513.00","1,480.00"
Max,"3,949.00","6,884.00"
N,"273,991.00","402,168.00"


In [23]:
## Combine into paper table (TU spells, max_TC_sample==2)
ger_tu = tab_ger_dur["days_if_TU"]
spa_tu = tab_spa_dur["days_if_TU"]

rows = {
    "Mean (days)":    [ger_mu_y,             spa_mu_y],
    "Variance":       [ger_tu["SD"]**2,       spa_tu["SD"]**2],
    "Log (variance)": [ger_lsig_y2,           spa_lsig_y2],
    "Min":            [ger_tu["Min"],          spa_tu["Min"]],
    "p10":            [ger_tu["p10"],          spa_tu["p10"]],
    "p25":            [ger_tu["p25"],          spa_tu["p25"]],
    "p50":            [ger_tu["p50"],          spa_tu["p50"]],
    "p75":            [ger_tu["p75"],          spa_tu["p75"]],
    "p90":            [ger_tu["p90"],          spa_tu["p90"]],
    "Max":            [ger_tu["Max"],          spa_tu["Max"]],
    "N":              [ger_N_ind,              spa_N_ind],
}
tab_dur = pd.DataFrame(rows, index=["Germany", "Spain"]).T
# print(tab_dur.to_latex(escape=False, float_format="%.3f"))
tab_dur


,Germany,Spain
Mean (days),360.18,912.45
Variance,"414,134.85","923,454.50"
Log (variance),2.57,1.86
Min,0.00,1.00
p10,29.00,59.00
p25,88.00,186.00
p50,317.00,483.00
p75,819.00,"1,075.00"
p90,"1,480.00","1,946.00"
Max,"6,884.00","10,265.00"


<a id="main"></a>

### 1. Main descriptive statistics, individual level (`tab:desstats_2+spell`)


In [24]:
tab_ger_main = pd.read_csv("./output/intermediate/descriptive_stats_ger_tab1.csv",index_col=0)
stats_names = ["Mean", "SD", "Min", "p10", "p25", "p50", "p75", "p90", "Max", "N"]
# stats_index = stats_names *3 + ["Total",]
stats_index = [f"{stat} {i}" for i in range(3) for stat in stats_names] + ["Total",]
tab_ger_main.index = stats_index
tab_ger_main = tab_ger_main.T
tab_ger_main = tab_ger_main.rename(index={"foreig~n": "foreign"})
tab_ger_main

,Mean 0,SD 0,Min 0,p10 0,p25 0,p50 0,p75 0,p90 0,Max 0,N 0,Mean 1,SD 1,Min 1,p10 1,p25 1,p50 1,p75 1,p90 1,Max 1,N 1,Mean 2,SD 2,Min 2,p10 2,p25 2,p50 2,p75 2,p90 2,Max 2,N 2,Total
age,36.92,9.63,25.00,26.00,28.00,35.00,46.00,51.00,54.00,"263,226.00",35.09,8.17,25.00,27.00,29.00,32.00,40.00,49.00,54.00,"104,883.00",36.60,7.17,25.00,29.00,31.00,35.00,40.00,49.00,54.00,"81,768.00",36.44
sex_d,0.42,0.49,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"263,226.00",0.44,0.50,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"104,883.00",0.50,0.50,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"81,768.00",0.44
foreign,0.33,0.47,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"263,226.00",0.28,0.45,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"104,883.00",0.26,0.44,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"81,768.00",0.30
n_PC,0.60,0.49,0.00,0.00,0.00,1.00,1.00,1.00,1.00,"263,226.00",1.42,0.49,0.00,1.00,1.00,1.00,2.00,2.00,2.00,"104,883.00",3.35,1.44,1.00,2.00,2.00,3.00,4.00,5.00,31.00,"81,768.00",1.29
n_TC_s~e,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"263,226.00",1.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,"104,883.00",2.85,1.35,2.00,2.00,2.00,2.00,3.00,4.00,30.00,"81,768.00",0.75
days_p~l,"4,063.91","5,059.94",0.00,0.00,0.00,730.00,"8,766.00","12,194.00","17,166.00","263,226.00","2,697.70","3,271.33",1.00,133.00,424.00,"1,309.00","3,614.00","8,123.00","16,733.00","104,883.00","3,086.14","2,952.32",3.00,454.00,980.00,"2,027.00","4,228.50","7,535.00","17,010.00","81,768.00","3,567.68"
days_tu,513.47,924.61,0.00,0.00,0.00,0.00,592.00,"1,947.00","7,471.00","263,226.00",852.26,890.21,0.00,46.00,141.00,518.00,"1,307.00","2,208.00","7,057.00","104,883.00","1,016.06",797.92,2.00,153.00,383.00,820.00,"1,476.00","2,175.00","7,301.00","81,768.00",683.80


In [25]:
# tab_spa_main = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab1.csv",index_col=0).T
tab_spa_main = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab_sample1.csv",index_col=0).T
tab_spa_main

max_TC_sample_classic,0,,,,,,,,,,1,,,,,,,,,,2,,,,,,,,,,Total
age,34.62,8.09,25.00,26.00,28.00,32.00,40.00,47.00,54.00,"312,629.00",40.45,8.16,25.00,29.00,34.00,41.00,47.00,52.00,54.00,"184,221.00",41.35,7.37,25.00,31.00,36.00,41.00,47.00,52.00,54.00,"69,406.00",37.34
sex_d,0.49,0.50,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"312,629.00",0.47,0.50,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"184,221.00",0.43,0.49,0.00,0.00,0.00,0.00,1.00,1.00,1.00,"69,406.00",0.48
foreign,0.13,0.33,0.00,0.00,0.00,0.00,0.00,1.00,1.00,"312,629.00",0.14,0.35,0.00,0.00,0.00,0.00,0.00,1.00,1.00,"184,221.00",0.18,0.38,0.00,0.00,0.00,0.00,0.00,1.00,1.00,"69,406.00",0.14
n_PC,1.48,1.61,0.00,0.00,1.00,1.00,2.00,3.00,74.00,"312,629.00",2.06,1.54,0.00,1.00,1.00,2.00,2.00,4.00,60.00,"184,221.00",3.52,1.86,0.00,2.00,2.00,3.00,4.00,6.00,114.00,"69,406.00",1.92
n_TC_s~e,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"121,256.00",1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,"56,630.00",2.50,1.37,2.00,2.00,2.00,2.00,3.00,3.00,100.00,"21,008.00",0.55
days_p~l,"3,808.95","3,576.76",0.00,0.00,341.00,"3,056.00","6,441.00","8,761.00","22,687.00","312,629.00","2,550.11","2,234.97",0.00,186.00,778.00,"2,000.00","3,738.00","5,636.00","14,212.00","184,221.00","2,411.92","1,783.04",0.00,528.00,"1,072.00","2,008.00","3,320.00","4,871.00","12,504.00","69,406.00","3,228.18"
days_tu,"1,983.23","2,739.32",0.00,0.00,0.00,728.00,"2,996.00","6,088.00","20,232.00","312,629.00","2,754.16","2,446.83",-166.00,273.00,765.00,"2,064.00","4,123.00","6,347.00","15,487.00","184,221.00","2,733.20","1,881.76",4.00,687.00,"1,278.00","2,335.00","3,766.00","5,328.00","11,590.00","69,406.00","2,325.97"


In [26]:
## -- Whole-sample descriptive statistics, incl. never-unstable (max_TC_sample 0/1/2) --
## Germany: Table 9 of 05_descriptives.log (Jenny's windowed-truncation fix, 2026-07-08)
##          -> Results/descriptive_stats_ger_tab8.csv (parser's sequential numbering
##          for this table shifted after the do-file restructure; verify against the
##          log if it drifts again — do-file's own "Table 9" comment is the source of truth).
## Spain:   descriptive_stats_spa_tab_sample1.csv (group 0 = never unstable, included) — UNCHANGED
## Both countries now key on 'days_p~l' (days_p_total, truncated by Stata) for T_S.

tab_ger_whole = pd.read_csv("./output/intermediate/descriptive_stats_ger_tab8.csv", index_col=0)
tab_ger_whole.index = stats_index
tab_ger_whole = tab_ger_whole.T
tab_ger_whole = tab_ger_whole.rename(index={"foreig~n": "foreign"})

# Rebuild tab_spa_main here too (not just in the later cell) so this cell
# does not depend on cell execution order -- cell 13 only does a naive
# .T without stats_index, and the "real" build was in a later cell.
tab_spa_main = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab_sample1.csv", index_col=0)
tab_spa_main.index = stats_index
tab_spa_main = tab_spa_main.T

_VARS = ['age', 'sex_d', 'foreign', 'n_PC', 'n_TC_s~e', 'days_p~l', 'days_tu']
_COLS = ['Age', 'Female', 'Non-national', r'$N_S$', r'$N_U$', r'$T_S$', r'$T_U$']

def _whole_df(tab, country):
    """Mean / Median / SD per group (0,1,2) for one country; days -> years."""
    recs = {}
    for i in range(3):
        for stat, key in [("Mean", "Mean"), ("Median", "p50"), ("SD", "SD")]:
            vals = [float(tab.loc[v, f'{key} {i}']) for v in _VARS]
            vals[5] /= 365.0   # T_S : days -> years
            vals[6] /= 365.0   # T_U : days -> years
            recs[(country, i, stat)] = vals
    return pd.DataFrame(recs, index=_COLS).T

table_whole = pd.concat([_whole_df(tab_ger_whole, "Germany"),
                         _whole_df(tab_spa_main, "Spain")])

# LaTeX (Mean / Median / (SD) per group), matching the paper layout
_colspec = "l" * (3 + len(_COLS))
_lines = [rf"\begin{{tabular}}{{{_colspec}}}", r"\toprule",
          r" &  &  & " + " & ".join(_COLS) + r" \\", r"\midrule"]
for ci, country in enumerate(["Germany", "Spain"]):
    if ci:
        _lines.append(r"\midrule")
    for i in range(3):
        mean = table_whole.loc[(country, i, "Mean")]
        med  = table_whole.loc[(country, i, "Median")]
        sd   = table_whole.loc[(country, i, "SD")]
        head = country if i == 0 else ""
        _lines.append(f"{head} & {i} & Mean & " + " & ".join(f"{v:.2f}" for v in mean) + r" \\")
        _lines.append(r" &  & Median & "        + " & ".join(f"{v:.2f}" for v in med)  + r" \\")
        _lines.append(r" &  &  & "              + " & ".join(f"({np.round(v, 2)})" for v in sd) + r" \\")
_lines += [r"\bottomrule", r"\end{tabular}"]
tex_whole_out = "\n".join(_lines)
print(tex_whole_out)

table_whole

\begin{tabular}{llllllllll}
\toprule
 &  &  & Age & Female & Non-national & $N_S$ & $N_U$ & $T_S$ & $T_U$ \\
\midrule
Germany & 0 & Mean & 40.63 & 0.48 & 0.23 & 0.51 & 0.00 & 12.79 & 7.37 \\
 &  & Median & 42.00 & 0.00 & 0.00 & 1.00 & 0.00 & 8.35 & 3.72 \\
 &  &  & (9.56) & (0.5) & (0.42) & (0.5) & (0.0) & (12.96) & (8.39) \\
 & 1 & Mean & 40.29 & 0.48 & 0.19 & 1.30 & 1.00 & 12.55 & 7.90 \\
 &  & Median & 40.00 & 0.00 & 0.00 & 1.00 & 1.00 & 10.24 & 5.99 \\
 &  &  & (9.01) & (0.5) & (0.39) & (0.46) & (0.0) & (10.49) & (7.1) \\
 & 2 & Mean & 40.98 & 0.50 & 0.18 & 3.25 & 2.92 & 11.67 & 7.78 \\
 &  & Median & 40.00 & 0.00 & 0.00 & 3.00 & 2.00 & 9.90 & 5.92 \\
 &  &  & (7.93) & (0.5) & (0.39) & (1.5) & (1.44) & (8.5) & (6.54) \\
\midrule
Spain & 0 & Mean & 34.62 & 0.49 & 0.13 & 1.48 & 0.00 & 10.44 & 5.43 \\
 &  & Median & 32.00 & 0.00 & 0.00 & 1.00 & 0.00 & 8.37 & 1.99 \\
 &  &  & (8.09) & (0.5) & (0.33) & (1.61) & (0.0) & (9.8) & (7.5) \\
 & 1 & Mean & 40.45 & 0.47 & 0.14 & 2.06 & 1.00 & 6

Age  Female  Non-national  $N_S$  $N_U$  $T_S$  $T_U$
Germany 0 Mean   40.63    0.48          0.23   0.51   0.00  12.79   7.37
          Median 42.00    0.00          0.00   1.00   0.00   8.35   3.72
          SD      9.56    0.50          0.42   0.50   0.00  12.96   8.39
        1 Mean   40.29    0.48          0.19   1.30   1.00  12.55   7.90
          Median 40.00    0.00          0.00   1.00   1.00  10.24   5.99
          SD      9.01    0.50          0.39   0.46   0.00  10.49   7.10
        2 Mean   40.98    0.50          0.18   3.25   2.92  11.67   7.78
          Median 40.00    0.00          0.00   3.00   2.00   9.90   5.92
          SD      7.93    0.50          0.39   1.50   1.44   8.50   6.54
Spain   0 Mean   34.62    0.49          0.13   1.48   0.00  10.44   5.43
          Median 32.00    0.00          0.00   1.00   0.00   8.37   1.99
          SD      8.09    0.50          0.33   1.61   0.00   9.80   7.50
        1 Mean   40.45    0.47          0.14   2.06   1.00   6.99   7.55
          Median 41.00    0.00          0.00   2.00   1.00   5.48   5.65
          SD      8.16    0.50          0.35   1.54   0.00   6.12   6.70
        2 Mean   41.35    0.43          0.18   3.52   2.50   6.61   7.49
          Median 41.00    0.00          0.00   3.00   2.00   5.50   6.40
          SD      7.37    0.49          0.38   1.86   1.37   4.89   5.16

In [27]:
# tab_spa_main = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab1.csv",index_col=0).T
tab_spa_main = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab_sample1.csv",index_col=0)
tab_spa_main.index = stats_index
tab_spa_main = tab_spa_main.T

stats = ['Mean','SD','N']
tabs_ger, tabs_spa = [],[]
for i in range(3):
    cols_i = [f"{stat} {i}" for stat in stats]
    tab_ger_i, tab_spa_i = tab_ger_main.loc[:,cols_i].T.copy(), tab_spa_main.loc[:,cols_i].T.copy()
    tab_ger_i.index,tab_spa_i.index = ["Mean",  "", "N"], ["Mean", "","N"]
    tabs_ger.append(tab_ger_i.T)
    tabs_spa.append(tab_spa_i.T)

tab_ger = pd.concat(tabs_ger,axis=1,keys=range(3))
tab_spa = pd.concat(tabs_spa,axis=1,keys=range(3))
table_merged = pd.concat((tab_ger,tab_spa),axis=1,keys=['Germany','Spain']).T

# table_merged.loc[:,'days_p'] = table_merged.loc[:,'days_p']/365
# table_merged.loc[:,'days_tu'] = table_merged.loc[:,'days_tu']/365

colnames_fixed = ['Age', 'Female', 'Non-national', r'$N_S$', r"$N_U$", r"$T_S$",r"$T_U$"]
table_merged.rename(columns=dict(zip(list(table_merged.keys()),colnames_fixed)),inplace=True)

table_merged = table_merged[['Age', 'Female', 'Non-national', r'$N_S$', r"$N_U$"]]

# # table_merged.index = ["Mean", "Median", ""]*6
table_merged = table_merged.astype('object')
# for row in range(1,12,2):
#     table_merged.iloc[row,:] = table_merged.iloc[row,:].apply(lambda x: "("+str(np.round(x,2))+")")

table_merged

Age     Female Non-national      $N_S$      $N_U$
Germany 0 Mean      36.92       0.42         0.33       0.60       0.00
                     9.63       0.49         0.47       0.49       0.00
          N    263,226.00 263,226.00   263,226.00 263,226.00 263,226.00
        1 Mean      35.09       0.44         0.28       1.42       1.00
                     8.17       0.50         0.45       0.49       0.00
          N    104,883.00 104,883.00   104,883.00 104,883.00 104,883.00
        2 Mean      36.60       0.50         0.26       3.35       2.85
                     7.17       0.50         0.44       1.44       1.35
          N     81,768.00  81,768.00    81,768.00  81,768.00  81,768.00
Spain   0 Mean      34.62       0.49         0.13       1.48       0.00
                     8.09       0.50         0.33       1.61       0.00
          N    312,629.00 312,629.00   312,629.00 312,629.00 121,256.00
        1 Mean      40.45       0.47         0.14       2.06       1.00
                     8.16       0.50         0.35       1.54       0.00
          N    184,221.00 184,221.00   184,221.00 184,221.00  56,630.00
        2 Mean      41.35       0.43         0.18       3.52       2.50
                     7.37       0.49         0.38       1.86       1.37
          N     69,406.00  69,406.00    69,406.00  69,406.00  21,008.00

In [28]:
print(table_merged.to_latex(escape=False,float_format="%.2f"))

\begin{tabular}{llllllll}
\toprule
 &  &  & Age & Female & Non-national & $N_S$ & $N_U$ \\
\midrule
\multirow[t]{9}{*}{Germany} & \multirow[t]{3}{*}{0} & Mean & 36.92 & 0.42 & 0.33 & 0.60 & 0.00 \\
 &  &  & 9.63 & 0.49 & 0.47 & 0.49 & 0.00 \\
 &  & N & 263226.00 & 263226.00 & 263226.00 & 263226.00 & 263226.00 \\
\cline{2-8}
 & \multirow[t]{3}{*}{1} & Mean & 35.09 & 0.44 & 0.28 & 1.42 & 1.00 \\
 &  &  & 8.17 & 0.50 & 0.45 & 0.49 & 0.00 \\
 &  & N & 104883.00 & 104883.00 & 104883.00 & 104883.00 & 104883.00 \\
\cline{2-8}
 & \multirow[t]{3}{*}{2} & Mean & 36.60 & 0.50 & 0.26 & 3.35 & 2.85 \\
 &  &  & 7.17 & 0.50 & 0.44 & 1.44 & 1.35 \\
 &  & N & 81768.00 & 81768.00 & 81768.00 & 81768.00 & 81768.00 \\
\cline{1-8} \cline{2-8}
\multirow[t]{9}{*}{Spain} & \multirow[t]{3}{*}{0} & Mean & 34.62 & 0.49 & 0.13 & 1.48 & 0.00 \\
 &  &  & 8.09 & 0.50 & 0.33 & 1.61 & 0.00 \\
 &  & N & 312629.00 & 312629.00 & 312629.00 & 312629.00 & 121256.00 \\
\cline{2-8}
 & \multirow[t]{3}{*}{1} & Mean & 40.45 & 0.4

<a id="whole"></a>

### 1b. Whole sample including never-unstable (`tab:desstats_all`)

Same variables and layout as the main table, but on the **whole sample including
never-unstable individuals** (groups `max_TC_sample` = 0, 1, 2):

- **Germany:** Table 2 of `data/intermediate/germany/05_descriptives.log` -> `Results/descriptive_stats_ger_tab1.csv`
- **Spain:** `data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab_sample1.csv` (group 0 = never unstable, included)

*Note:* Spain's permanent-days column is `days_p~l` vs Germany's `days_p`; both map to $T_S$ (converted to years).

In [29]:
# SUPERSEDED (2026-07-23): this was an older, incomplete duplicate of the
# Table 2b build (only 3 vars, no T_S/T_U, used tab_ger_main instead of
# tab_ger_whole). It ran AFTER cell 14 and silently overwrote table_whole/
# tex_whole_out with the old version before the final export cell used them.
# Emptied out so cell 14's correct T_S/T_U version survives to export.


<a id="educ"></a>

### 2. Education breakdown (`tab:desstats_2+spell`)


In [30]:
tab_ger_educ = pd.read_csv("./output/intermediate/descriptive_stats_ger_tab4.csv",index_col=0)
# tab_ger_educ.loc[:,'Freq.'] = tab_ger_educ.loc[:,'Freq.'].apply(lambda x: x.replace(",","")).apply(pd.to_numeric)
# tab_ger_educ.loc[:,'Cum.'] = tab_ger_educ.loc[:,'Cum.'].apply(pd.to_numeric)

tab_ger_educ.loc["0",:] = tab_ger_educ.loc["0",:].apply(lambda x: x.replace(",",""))
tab_ger_educ = tab_ger_educ.astype(float)
tab_ger_educ.loc['Primary School',:] = tab_ger_educ.loc['Primary School',:]+tab_ger_educ.loc['0',:]
tab_ger_educ.iloc[3,:] = tab_ger_educ.iloc[3,:]+tab_ger_educ.iloc[1,:]
tab_ger_educ = tab_ger_educ.iloc[2:,:]

tab_spa_educ = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_other_sample1.csv",index_col=0)
for i in range(tab_spa_educ.shape[1]):
    tab_spa_educ.iloc[:,i] = tab_spa_educ.iloc[:,i].apply(lambda x: x.replace(",",""))
tab_spa_educ = tab_spa_educ.astype(float)

tab_ger_educ.head()

,0,1,2,Total
educ2,,,,
Primary School,"48,146.00","9,356.00","4,748.00","62,250.00"
,18.29,8.92,5.81,13.83
Lower Secondary Schoo,"61,409.00","21,029.00","16,437.00","98,875.00"
,23.33,20.05,20.10,21.98
Upper Secondary Schoo,"99,233.00","46,538.00","43,785.00","189,556.00"


In [31]:
pd.read_csv("./output/intermediate/descriptive_stats_ger_tab6.csv",index_col=0)

,Freq.,Percent,Cum.
starts_TU,,,
0,"365,754",41.92,41.92
1,"506,839",58.08,100.00
Total,872593,100.00,NaN


In [32]:
stats = ['Mean','SD']
tabs_ger, tabs_spa = [],[]
for i in range(3):
    cols_i = [f"{stat} {i}" for stat in stats]
    tab_ger_i, tab_spa_i = tab_ger_main.loc[:,cols_i].T.copy(), tab_spa_main.loc[:,cols_i].T.copy()
    tab_ger_i.index,tab_spa_i.index = ["Mean", ""], ["Mean", ""]
    tabs_ger.append(tab_ger_i.T)
    tabs_spa.append(tab_spa_i.T)

tab_ger = pd.concat(tabs_ger,axis=1,keys=range(3))
tab_spa = pd.concat(tabs_spa,axis=1,keys=range(3))
table_merged2 = pd.concat((tab_ger,tab_spa),axis=1,keys=['Germany','Spain']).T
table_merged2 = table_merged2.loc[:,["age","sex_d","foreign"]]

edu_names = list(tab_ger_educ.index[np.where(tab_ger_educ.index!=" ")[0]])[:-1]
for i, educ in enumerate(edu_names):
    uni_array = np.hstack((tab_ger_educ.iloc[i*2+1,:-1],tab_spa_educ.iloc[i*2+1,:-1]))
    uni_array = np.insert(uni_array, range(1, uni_array.size+1), 0)
    table_merged2[educ] = uni_array
table_merged2 = table_merged2.T

tab_new = table_merged2.iloc[:,np.arange(0,table_merged2.shape[1],2)].copy()
tab_std = table_merged2.iloc[:,np.arange(1,table_merged2.shape[1],2)].copy()
tab_new = tab_new.droplevel(2,axis=1)
tab_std = tab_std.droplevel(2,axis=1)
tab_all = pd.concat([tab_new.iloc[:1,:].T,tab_std.iloc[0,:]],axis=1)
var_list = np.array(table_merged2.index)
for i in range(1,var_list.size):
    tab_all = pd.concat([tab_all,tab_new.iloc[i,:].T,tab_std.iloc[i,:]],axis=1)
tab_all = tab_all.T
var_list_ext = np.array(['Age', 'Female', 'Non-national', "Primary", "Lower Secondary", "Upper Secondary", "College"])
tab_all.index = np.insert(var_list_ext, range(1, var_list_ext.size+1), "")

tab_all = tab_all.astype('object')
for row in range(1,tab_all.shape[0],2):
    tab_all.iloc[row,:] = tab_all.iloc[row,:].apply(lambda x: "("+str(np.round(x,2))+")")

tab_mean3 = table_merged.iloc[::3,-4:].T.copy()
tab_std3 = table_merged.iloc[2::3,-4:].T.copy()
tab_mean3 = tab_mean3.droplevel(2,axis=1)
tab_std3 = tab_std3.droplevel(2,axis=1)

tab_all = tab_all.T
var_list = np.array(tab_mean3.index)
for i in range(var_list.size):
    tab_all = pd.concat([tab_all,tab_mean3.iloc[i,:].T,tab_std3.iloc[i,:].T.rename("")],axis=1)
tab_all = tab_all.T

tab_all

Germany                           Spain             \
                         0          1         2          0          1   
Age                  36.92      35.09     36.60      34.62      40.45   
                    (9.63)     (8.17)    (7.17)     (8.09)     (8.16)   
Female                0.42       0.44      0.50       0.49       0.47   
                    (0.49)      (0.5)     (0.5)      (0.5)      (0.5)   
Non-national          0.33       0.28      0.26       0.13       0.14   
                    (0.47)     (0.45)    (0.44)     (0.33)     (0.35)   
Primary              18.29       8.92      5.81      15.36      14.57   
                     (0.0)      (0.0)     (0.0)      (0.0)      (0.0)   
Lower Secondary      23.33      20.05     20.10      35.54      38.57   
                     (0.0)      (0.0)     (0.0)      (0.0)      (0.0)   
Upper Secondary      37.70      44.37     53.55      24.88      24.42   
                     (0.0)      (0.0)     (0.0)      (0.0)      (0.0)   
College              20.68      26.66     20.54      24.22      22.43   
                     (0.0)      (0.0)     (0.0)      (0.0)      (0.0)   
Female                0.42       0.44      0.50       0.49       0.47   
                263,226.00 104,883.00 81,768.00 312,629.00 184,221.00   
Non-national          0.33       0.28      0.26       0.13       0.14   
                263,226.00 104,883.00 81,768.00 312,629.00 184,221.00   
$N_S$                 0.60       1.42      3.35       1.48       2.06   
                263,226.00 104,883.00 81,768.00 312,629.00 184,221.00   
$N_U$                 0.00       1.00      2.85       0.00       1.00   
                263,226.00 104,883.00 81,768.00 121,256.00  56,630.00   

                           
                        2  
Age                 41.35  
                   (7.37)  
Female               0.43  
                   (0.49)  
Non-national         0.18  
                   (0.38)  
Primary             12.16  
                    (0.0)  
Lower Secondary     39.62  
                    (0.0)  
Upper Secondary     26.22  
                    (0.0)  
College             22.00  
                    (0.0)  
Female               0.43  
                69,406.00  
Non-national         0.18  
                69,406.00  
$N_S$                3.52  
                69,406.00  
$N_U$                2.50  
                21,008.00

In [33]:
print(tab_all.to_latex(escape=False,float_format="%.2f"))

\begin{tabular}{lllllll}
\toprule
 & \multicolumn{3}{r}{Germany} & \multicolumn{3}{r}{Spain} \\
 & 0 & 1 & 2 & 0 & 1 & 2 \\
\midrule
Age & 36.92 & 35.09 & 36.60 & 34.62 & 40.45 & 41.35 \\
 & (9.63) & (8.17) & (7.17) & (8.09) & (8.16) & (7.37) \\
Female & 0.42 & 0.44 & 0.50 & 0.49 & 0.47 & 0.43 \\
 & (0.49) & (0.5) & (0.5) & (0.5) & (0.5) & (0.49) \\
Non-national & 0.33 & 0.28 & 0.26 & 0.13 & 0.14 & 0.18 \\
 & (0.47) & (0.45) & (0.44) & (0.33) & (0.35) & (0.38) \\
Primary & 18.29 & 8.92 & 5.81 & 15.36 & 14.57 & 12.16 \\
 & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) \\
Lower Secondary & 23.33 & 20.05 & 20.10 & 35.54 & 38.57 & 39.62 \\
 & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) \\
Upper Secondary & 37.70 & 44.37 & 53.55 & 24.88 & 24.42 & 26.22 \\
 & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) \\
College & 20.68 & 26.66 & 20.54 & 24.22 & 22.43 & 22.00 \\
 & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) & (0.0) \\
Female & 0.42 & 0.44 & 0.50 & 0.49 & 0.47 & 0.43 \\
 & 263226.00 & 104883.

In [34]:
edu_dict = dict(zip(['1','2','3','4'],['Primary School', 'Lower Secondary School', 'Upper Secondary School','University']))

# tab_spa_educ = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_tab2.csv",index_col=0,keep_default_na=False,skip_blank_lines=True)
tab_spa_educ = pd.read_csv("./data/intermediate/spain/descriptive_stats/descriptive_stats_spa_other_sample1.csv",index_col=0)
for i in range(tab_spa_educ.shape[1]):
    tab_spa_educ.iloc[:,i] = tab_spa_educ.iloc[:,i].apply(lambda x: x.replace(",",""))
tab_spa_educ = tab_spa_educ.astype(float)
# tab_spa_educ = tab_spa_educ.loc[["1","2","3","4"],:].copy()
tab_spa_educ.rename(index=edu_dict,inplace='True')
# tab_spa_educ.loc[:,:] = tab_spa_educ.loc[:,:].apply(lambda x: x.replace(",","")).apply(pd.to_numeric)
# tab_spa_educ.loc[:,'Cum.'] = tab_spa_educ.loc[:,'Cum.'].apply(pd.to_numeric)
# tab_spa_educ.loc['Total',:] = np.sum(tab_spa_educ.values,axis=0)
tab_spa_educ

,0,1,2,Total
educ2,,,,
Primary School,"46,514.00","26,278.00","8,240.00","81,032.00"
,15.36,14.57,12.16,14.71
Lower Secondary School,"107,645.00","69,570.00","26,847.00","204,062.00"
,35.54,38.57,39.62,37.04
Upper Secondary School,"75,343.00","44,047.00","17,766.00","137,156.00"
,24.88,24.42,26.22,24.89
University,"73,353.00","40,455.00","14,908.00","128,716.00"
,24.22,22.43,22.00,23.36
Total,"302,855.00","180,350.00","67,761.00","550,966.00"


In [35]:
tab_ger_educ

,0,1,2,Total
educ2,,,,
Primary School,"48,146.00","9,356.00","4,748.00","62,250.00"
,18.29,8.92,5.81,13.83
Lower Secondary Schoo,"61,409.00","21,029.00","16,437.00","98,875.00"
,23.33,20.05,20.10,21.98
Upper Secondary Schoo,"99,233.00","46,538.00","43,785.00","189,556.00"
,37.70,44.37,53.55,42.14
University,"54,438.00","27,960.00","16,798.00","99,196.00"
,20.68,26.66,20.54,22.05
Total,"263,226.00","104,883.00","81,768.00","449,877.00"


In [36]:
table_educ = pd.concat(
    (tab_ger_educ.reset_index(drop=True), tab_spa_educ.reset_index(drop=True)),
    axis=1, keys=['Germany','Spain']
)
table_educ.index = list(tab_ger_educ.index)

table_educ
print(table_educ.to_latex(escape=False,float_format="%.2f"))

\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{r}{Germany} & \multicolumn{4}{r}{Spain} \\
 & 0 & 1 & 2 & Total & 0 & 1 & 2 & Total \\
\midrule
Primary School & 48146.00 & 9356.00 & 4748.00 & 62250.00 & 46514.00 & 26278.00 & 8240.00 & 81032.00 \\
  & 18.29 & 8.92 & 5.81 & 13.83 & 15.36 & 14.57 & 12.16 & 14.71 \\
Lower Secondary Schoo & 61409.00 & 21029.00 & 16437.00 & 98875.00 & 107645.00 & 69570.00 & 26847.00 & 204062.00 \\
  & 23.33 & 20.05 & 20.10 & 21.98 & 35.54 & 38.57 & 39.62 & 37.04 \\
Upper Secondary Schoo & 99233.00 & 46538.00 & 43785.00 & 189556.00 & 75343.00 & 44047.00 & 17766.00 & 137156.00 \\
  & 37.70 & 44.37 & 53.55 & 42.14 & 24.88 & 24.42 & 26.22 & 24.89 \\
University & 54438.00 & 27960.00 & 16798.00 & 99196.00 & 73353.00 & 40455.00 & 14908.00 & 128716.00 \\
  & 20.68 & 26.66 & 20.54 & 22.05 & 24.22 & 22.43 & 22.00 & 23.36 \\
Total & 263226.00 & 104883.00 & 81768.00 & 449877.00 & 302855.00 & 180350.00 & 67761.00 & 550966.00 \\
\bottomrule
\end{tabular}



In [37]:
## ── Industry of last employment: parse log files ──────────────────────────

def parse_ind_tab_log(log_path, start_marker):
    """
    Parse a Stata 'tab ind_short max_TC_sample [if ...], column' output block.

    Each industry is printed as two alternating lines:
      1. Frequency row : <label text>  <n0>  <n1>  <n2>  <n_total>
      2. Percentage row:               <p0>  <p1>  <p2>  <p_total>  (blank label)

    To correctly handle labels like 'Manufacturing 1' (where the number is
    part of the label, not a count), we split by taking the *last 4 tokens*
    as the numeric values and everything before them as the label.

    Returns:
        df_pct  : DataFrame of column-% shares; rows=industry labels,
                  columns=['No TC', '1 TC', '2+ TC', 'Total']
        n_total : list [N_0, N_1, N_2, N_total] from the Total frequency row
    """

    def _split_label_nums(words):
        """Take last 4 convertible tokens as numbers; rest is the label."""
        nums       = []
        label_parts = list(words)
        while label_parts and len(nums) < 4:
            try:
                nums.insert(0, float(label_parts[-1].replace(',', '')))
                label_parts.pop()
            except ValueError:
                break  # non-numeric token reached
        return label_parts, nums

    reading       = False
    rows          = []
    waiting_pct   = False
    current_label = None
    current_freq  = None

    with open(log_path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    for line in lines:
        if not reading:
            if start_marker in line:
                reading = True
            continue

        clean    = line.replace('|', ' ').rstrip('\n')
        stripped = clean.strip()
        if not stripped:
            continue
        # Skip decorative separator lines
        if all(c in '-+| \t' for c in line.rstrip('\n')):
            continue

        words = stripped.split()
        if not words:
            continue
        # Skip header / log-open / log-close lines
        if words[0] in ['Key', 'frequency', 'column', 'max_TC_sample',
                         'Industry', 'ind_short', 'name:', 'log:', 'opened', 'closed']:
            continue

        label_parts, nums = _split_label_nums(words)
        if len(nums) < 4:
            continue  # malformed line

        if label_parts:
            # Non-empty label → frequency row
            current_label = ' '.join(label_parts)
            current_freq  = nums
            waiting_pct   = True
        elif waiting_pct and current_label is not None:
            # Empty label → percentage row following the previous frequency row
            rows.append({'label': current_label, 'freq': current_freq, 'pct': nums})
            waiting_pct   = False
            current_label = None
            current_freq  = None
            if rows[-1]['label'] == 'Total':
                break  # end of table

    total_row = next((r for r in rows if r['label'] == 'Total'), None)
    ind_rows  = [r for r in rows if r['label'] != 'Total']

    cols   = ['No TC', '1 TC', '2+ TC', 'Total']
    df_pct = pd.DataFrame(
        [r['pct'][:4] for r in ind_rows],
        index=[r['label'] for r in ind_rows],
        columns=cols
    )
    df_freq = pd.DataFrame(
        [r['freq'][:4] for r in ind_rows],
        index=[r['label'] for r in ind_rows],
        columns=cols
    ).astype(int)
    n_total = [int(x) for x in total_row['freq'][:4]] if total_row else [None] * 4
    return df_pct, df_freq, n_total


# Germany: individual-level table restricted to samesame==1 (one observation per person)
tab_ger_ind, freq_ger_ind, n_ger_ind = parse_ind_tab_log(
    res_ger_file,
    start_marker='tab ind_short max_TC_sample if samesame==1, column'
)

# Spain: standalone log file – trigger on the column-header line
tab_spa_ind, freq_spa_ind, n_spa_ind = parse_ind_tab_log(
    './data/intermediate/spain/descriptive_stats/desstats_sple_id_ind2011_short.log',
    start_marker='ind_short'
)

# ── Rename truncated Stata labels to clean full names ─────────────────────
# Germany: based on WZ 2008 codes in germany/01_clean.do
ger_label_map = {
    'NA':                    'Missing',
    'Agriculture':           'Agriculture',
    'Manufacturing 1':       'Manufacturing (consumer goods)',
    'Manufacturing 2':       r'Manufacturing (chemicals \& metals)',
    'Manufacturing 3':       r'Manufacturing (machinery \& electronics)',
    'Construction':          'Construction',
    'Wholesale and Retail':  'Wholesale and Retail Trade',
    'Transportation and St': 'Transportation and Storage',
    'Hospitality':           'Hospitality',
    'Business Services':     'Business Services',
    'Financial and Insuran': 'Financial and Insurance',
    'Public Admin and Defe': r'Public Admin.\ and Defence',
    'Education':             'Education',
    'Health Services':       'Health Services',
}

# Spain: based on CNAE 2009 codes in spain/format_files/industry_clean.do
spa_label_map = {
    'NA':                    'Missing',
    'Agriculture':           'Agriculture',
    'Extractive':            'Mining and Quarrying',
    'Manufactures (primary': 'Manufacturing (primary)',
    'Manufactures (machine': 'Manufacturing (machinery)',
    'Manufactures (detail':  'Manufacturing (detail and repair)',
    'Energy, gas, residual': 'Energy, Gas and Waste Treatment',
    'Construction':          'Construction',
    'Retail and repairs':    'Retail and Repairs',
    'Transport and storage': 'Transport and Storage',
    'Hospitality':           'Hospitality',
    'Communication and pro': 'Communication and Programming',
    'Financial':             'Financial Activities',
    'Real State':            'Real Estate',
    'Professional and scie': 'Professional and Scientific Services',
    'Auxiliary services (c': r'Auxiliary Services (cleaning, security)',
    'Public Administration': 'Public Administration',
    'Education':             'Education',
    'Health and Social Ser': 'Health and Social Services',
    'Other Services':        'Other Services',
}

tab_ger_ind.index  = [ger_label_map.get(l, l) for l in tab_ger_ind.index]
freq_ger_ind.index = tab_ger_ind.index
tab_spa_ind.index  = [spa_label_map.get(l, l) for l in tab_spa_ind.index]
freq_spa_ind.index = tab_spa_ind.index

print('Germany industry (% column share):')
display(tab_ger_ind)
print('\nSpain industry (% column share):')
display(tab_spa_ind)


Germany industry (% column share):


,No TC,1 TC,2+ TC,Total
Missing,14.21,8.29,9.17,11.91
Agriculture,3.28,1.97,2.03,2.75
Manufacturing (consumer goods),4.22,4.40,4.05,4.23
3,5.11,4.92,4.66,4.98
Manufacturing (machinery \& electronics),7.74,6.14,5.65,6.99
Construction,4.75,5.46,6.14,5.17
Wholesale and Retail Trade,9.98,11.78,11.56,10.69
Transportation and Storage,4.45,5.37,5.98,4.94
Hospitality,4.56,4.40,4.52,4.52
Business Services,17.42,23.38,20.94,19.45



Spain industry (% column share):


,No TC,1 TC,2+ TC,Total
Missing,2.45,4.88,0.78,3.34
Agriculture,6.84,2.51,1.68,3.75
Mining and Quarrying,0.31,0.20,0.11,0.22
Manufacturing (primary),2.84,4.28,3.87,3.74
Manufacturing (machinery),2.77,5.88,4.97,4.71
Manufacturing (detail and repair),0.83,1.56,1.58,1.33
"Energy, Gas and Waste Treatment",1.20,1.17,0.77,1.11
Construction,10.75,8.68,9.52,9.50
Retail and Repairs,10.57,16.42,19.26,15.05
Transport and Storage,3.29,5.16,7.82,5.05


In [38]:
## ── Generate LaTeX for industry tables (main sample: max_TC_sample == 2) ────
#
# Two columns per country: count (N) and column % share for max_TC_sample == 2.

def _ind_tex(df_pct, df_freq, n_total):
    """Format count + % share for the 2+ TC group as a LaTeX tabular."""
    lines_ind = [
        r'\begin{tabular}{lrr}',
        r'\toprule',
        r'Industry & $N$ & Share (\%) \\',
        r'\midrule',
    ]
    for label in df_pct.index:
        n   = int(df_freq.loc[label, '2+ TC'])
        pct = df_pct.loc[label, '2+ TC']
        lines_ind.append(f"{label} & {n:,} & {pct:.2f} \\\\")

    lines_ind.append(r'\midrule')
    lines_ind.append(f"Total & {int(n_total[2]):,} & 100.00 \\\\")
    lines_ind += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines_ind)


tex_ger_ind = _ind_tex(tab_ger_ind, freq_ger_ind, n_ger_ind)
tex_spa_ind = _ind_tex(tab_spa_ind, freq_spa_ind, n_spa_ind)

print('% Table 3: Germany industry (max_TC_sample==2)')
print(tex_ger_ind)
print()
print('% Table 4: Spain industry (max_TC_sample==2)')
print(tex_spa_ind)


% Table 3: Germany industry (max_TC_sample==2)
\begin{tabular}{lrr}
\toprule
Industry & $N$ & Share (\%) \\
\midrule
Missing & 7,500 & 9.17 \\
Agriculture & 1,663 & 2.03 \\
Manufacturing (consumer goods) & 3,309 & 4.05 \\
3 & 3,811 & 4.66 \\
Manufacturing (machinery \& electronics) & 4,622 & 5.65 \\
Construction & 5,019 & 6.14 \\
Wholesale and Retail Trade & 9,453 & 11.56 \\
Transportation and Storage & 4,888 & 5.98 \\
Hospitality & 3,698 & 4.52 \\
Business Services & 17,126 & 20.94 \\
Financial and Insurance & 2,092 & 2.56 \\
Public Admin.\ and Defence & 3,809 & 4.66 \\
Education & 2,322 & 2.84 \\
Health Services & 12,456 & 15.23 \\
\midrule
Total & 81,768 & 100.00 \\
\bottomrule
\end{tabular}

% Table 4: Spain industry (max_TC_sample==2)
\begin{tabular}{lrr}
\toprule
Industry & $N$ & Share (\%) \\
\midrule
Missing & 538 & 0.78 \\
Agriculture & 1,167 & 1.68 \\
Mining and Quarrying & 73 & 0.11 \\
Manufacturing (primary) & 2,689 & 3.87 \\
Manufacturing (machinery) & 3,450 & 4.97 \\
Manu

---
## Export: write both tables to `output/tables/descriptive_stats_tables.tex`

In [39]:
## ── Table 1: Duration of unstable spells (tab:desstats_dur) ─────────────────
#
# Row formatting rules (matching paper):
#   Mean (days)    : 2 decimal places
#   Variance       : comma-separated, 1 decimal place
#   Log (variance) : 3 decimal places
#   Min/pXX/Max    : integer, comma-separated
#   N              : integer, comma-separated

def _fmt_dur_row(label, g, s):
    if label in ("Min", "p10", "p25", "p50", "p75", "p90", "Max"):
        suffix = r"\\"
        return f"{label} & {int(g):,} & {int(s):,} {suffix}"
    elif label == "N":
        return f"N & {int(g):,} & {int(s):,} \\\\[5pt]"
    elif label == "Mean (days)":
        return f"{label} & {g:,.2f} & {s:,.2f} \\\\"
    elif label == "Variance":
        return f"{label} & {g:,.1f} & {s:,.1f} \\\\"
    elif label == "Log (variance)":
        return f"{label} & {g:.3f} & {s:.3f} \\\\"

stat_order_dur = [
    "Mean (days)", "Variance", "Log (variance)",
    "Min", "p10", "p25", "p50", "p75", "p90", "Max", "N",
]

lines_dur = [
    r"\begin{tabular}{lll}",
    r"\toprule",
    r" & Germany & Spain \\",
    r"\midrule",
]
for stat in stat_order_dur:
    lines_dur.append(_fmt_dur_row(stat,
                                  tab_dur.loc[stat, 'Germany'],
                                  tab_dur.loc[stat, 'Spain']))
lines_dur += [r"\bottomrule", r"\end{tabular}"]

tex_dur_out = "\n".join(lines_dur)
print(tex_dur_out)

\begin{tabular}{lll}
\toprule
 & Germany & Spain \\
\midrule
Mean (days) & 360.18 & 912.45 \\
Variance & 414,134.9 & 923,454.5 \\
Log (variance) & 2.568 & 1.855 \\
Min & 0 & 1 \\
p10 & 29 & 59 \\
p25 & 88 & 186 \\
p50 & 317 & 483 \\
p75 & 819 & 1,075 \\
p90 & 1,480 & 1,946 \\
Max & 6,884 & 10,265 \\
N & 81,768 & 69,406 \\[5pt]
\bottomrule
\end{tabular}


In [40]:
## ── Table 2: Main descriptive statistics (tab:desstats_2+spell) ─────────────
#
# Shows only max_TC_sample == 2 (the paper's main sample).
# Continuous vars: Mean (2 dp) + (SD) below.
# Education: percentage (2 dp) from tab_ger_educ / tab_spa_educ,
#            group-2 column = positional index 2, percentage row = i*2+1.
# N_S, N_U : Mean (2 dp) + (SD) below.
# N        : integer, comma-separated.

def _mfmt(v):   return f"{float(v):.2f}"
def _sdfmt(v):  return f"({float(v):.2f})"

lines_main = [
    r"\begin{tabular}{lcc}",
    r"\toprule",
    r" & Germany & Spain \\",
    r"\midrule",
]


# ---- Continuous: Age, Female, Non-national ----
for label, var in [('Age', 'age'), ('Female', 'sex_d'), ('Non-national', 'foreign')]:
    gm = tab_ger_main.loc[var, 'Mean 2'];  gs = tab_ger_main.loc[var, 'SD 2']
    sm = tab_spa_main.loc[var, 'Mean 2'];  ss = tab_spa_main.loc[var, 'SD 2']
    lines_main.append(f"{label} & {_mfmt(gm)} & {_mfmt(sm)} \\\\")
    lines_main.append(f"    & {_sdfmt(gs)} & {_sdfmt(ss)} \\\\")

# ---- Education shares (%) ----
# tab_ger_educ / tab_spa_educ: alternating rows count / pct.
# After tab_ger_educ = tab_ger_educ.iloc[2:], order is:
#   iloc[0] Primary counts, iloc[1] Primary pct,
#   iloc[2] Lower Sec counts, iloc[3] Lower Sec pct, ...
# Column positional index 2  = group max_TC_sample == 2.
for i, label in enumerate(['Primary', 'Lower Secondary', 'Upper Secondary', 'College']):
    g_pct = tab_ger_educ.iloc[i * 2 + 1, 2]
    s_pct = tab_spa_educ.iloc[i * 2 + 1, 2]
    lines_main.append(f"{label} & {g_pct:.2f} & {s_pct:.2f} \\\\")
    lines_main.append(f"    & (0.00) & (0.00) \\\\")

lines_main.append(r"\midrule")

# ---- N_S and N_U ----
for label, var in [
    (r"$N_S$: number stable spells",   'n_PC'),
    (r"$N_U$: number unstable spells", 'n_TC_s~e'),
]:
    gm = tab_ger_main.loc[var, 'Mean 2'];  gs = tab_ger_main.loc[var, 'SD 2']
    sm = tab_spa_main.loc[var, 'Mean 2'];  ss = tab_spa_main.loc[var, 'SD 2']
    lines_main.append(f"{label} & {_mfmt(gm)} & {_mfmt(sm)} \\\\")
    lines_main.append(f"   & {_sdfmt(gs)} & {_sdfmt(ss)} \\\\")

lines_main.append(r"\hline \hline")

# ---- N (total individual count) ----
g_N = int(tab_ger_main.loc['age', 'N 2'])
s_N = int(tab_spa_main.loc['age', 'N 2'])
lines_main.append(f"$N$ & {g_N:,} & {s_N:,} \\\\")

lines_main += [r"\bottomrule", r"\end{tabular}"]

tex_main_out = "\n".join(lines_main)
print(tex_main_out)

\begin{tabular}{lcc}
\toprule
 & Germany & Spain \\
\midrule
Age & 36.60 & 41.35 \\
    & (7.17) & (7.37) \\
Female & 0.50 & 0.43 \\
    & (0.50) & (0.49) \\
Non-national & 0.26 & 0.18 \\
    & (0.44) & (0.38) \\
Primary & 5.81 & 12.16 \\
    & (0.00) & (0.00) \\
Lower Secondary & 20.10 & 39.62 \\
    & (0.00) & (0.00) \\
Upper Secondary & 53.55 & 26.22 \\
    & (0.00) & (0.00) \\
College & 20.54 & 22.00 \\
    & (0.00) & (0.00) \\
\midrule
$N_S$: number stable spells & 3.35 & 3.52 \\
   & (1.44) & (1.86) \\
$N_U$: number unstable spells & 2.85 & 2.50 \\
   & (1.35) & (1.37) \\
\hline \hline
$N$ & 81,768 & 69,406 \\
\bottomrule
\end{tabular}


In [41]:
## ── Write all tables to output/tables/descriptive_stats_tables.tex ───────────────────
output_tex = "./output/tables/descriptive_stats_tables.tex"

with open(output_tex, 'w', encoding='utf-8') as f:
    f.write("% Descriptive Statistics Tables\n")
    f.write("% Auto-generated by 'Descriptive Stats Tables.ipynb'\n")
    f.write("% Do not edit manually - re-run the notebook to update.\n\n")
    f.write("% Table 1: Duration of unstable spells (tab:desstats_dur)\n")
    f.write(tex_dur_out)
    f.write("\n\n")
    f.write("% Table 2: Descriptive statistics (tab:desstats_2+spell)\n")
    f.write(tex_main_out)
    f.write("\n\n")
    f.write("% Table 2b: Whole sample incl. never-unstable (tab:desstats_all)\n")
    f.write(tex_whole_out)
    f.write("\n\n")
    f.write("% Table 3: Industry of last employment -- Germany\n")
    f.write(tex_ger_ind)
    f.write("\n\n")
    f.write("% Table 4: Industry of last employment -- Spain\n")
    f.write(tex_spa_ind)
    f.write("\n")

print(f"Saved: {output_tex}")


Saved: ./output/tables/descriptive_stats_tables.tex
